<a href="https://colab.research.google.com/github/aclaire10/Datasci-112-Final-Project/blob/main/Datasci_112_Final_Project_Web_Scraping_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Web Scraping Code

This was the code we used to scrape the web for new datacenter developments and planned developments in order to curate one of our datasets. We utilized SerpAPI keys and OpenAI API keys to do it, with a delay to not overwhelm anything and make it as ethical as possible. This was implemented with python, and the adapted code with the necessary dependencies was pasted below, except for the API keys we used.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from lxml import html
from newspaper import Article
import lxml_html_clean

In [ ]:
import os
os.environ["SERPAPI_API_KEY"] = "YOUR_SERPAPI_KEY"
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"
# optional:
os.environ["OPENAI_MODEL"] = "gpt-4.1"

In [ ]:
#!/usr/bin/env python3
"""
scrape_new_data_centers_pipeline.py

Standalone pipeline to build a dataset of recent/planned U.S. data center developments
from verifiable web sources.

What it does:
  1) Uses SerpAPI (Google engine) to collect article URLs for a set of queries
  2) Scrapes article text (newspaper3k; fallback to BeautifulSoup)
  3) Uses OpenAI Chat Completions to STRICTLY extract structured fields (null if not explicit)
  4) Writes:
       - data/raw_articles.jsonl
       - data/dropped_rows.jsonl
       - data/new_data_centers.csv

Requirements (env vars):
  SERPAPI_API_KEY="..."
  OPENAI_API_KEY="..."
Optional:
  OPENAI_MODEL="gpt-4.1"
  OPENAI_FALLBACK_MODEL="gpt-4.1"

Install deps:
  pip install pandas requests beautifulsoup4 lxml newspaper3k lxml_html_clean
"""

from __future__ import annotations

import argparse
import csv
import datetime as dt
import json
import os
import random
import re
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    from newspaper import Article  # type: ignore
except Exception:
    Article = None


Status = Literal["announced", "under_construction", "opened"]

US_STATE_ABBREV: set[str] = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME","MD","MA","MI","MN",
    "MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA",
    "WV","WI","WY",
}

US_STATE_NAME_TO_ABBREV: dict[str, str] = {
    "alabama":"AL","alaska":"AK","arizona":"AZ","arkansas":"AR","california":"CA","colorado":"CO","connecticut":"CT","delaware":"DE",
    "district of columbia":"DC","florida":"FL","georgia":"GA","hawaii":"HI","idaho":"ID","illinois":"IL","indiana":"IN","iowa":"IA",
    "kansas":"KS","kentucky":"KY","louisiana":"LA","maine":"ME","maryland":"MD","massachusetts":"MA","michigan":"MI","minnesota":"MN",
    "mississippi":"MS","missouri":"MO","montana":"MT","nebraska":"NE","nevada":"NV","new hampshire":"NH","new jersey":"NJ","new mexico":"NM",
    "new york":"NY","north carolina":"NC","north dakota":"ND","ohio":"OH","oklahoma":"OK","oregon":"OR","pennsylvania":"PA","rhode island":"RI",
    "south carolina":"SC","south dakota":"SD","tennessee":"TN","texas":"TX","utah":"UT","vermont":"VT","virginia":"VA","washington":"WA",
    "west virginia":"WV","wisconsin":"WI","wyoming":"WY",
}

YEAR_MIN = 2018
YEAR_MAX = 2025

DEFAULT_QUERIES = [
    "data center announced USA 2024 city state",
    "data center under construction USA 2024 city state",
    "data center opened USA 2024 city state",
    "AWS data center announced USA 2024",
    "Microsoft Azure data center region announced USA 2024",
    "Google data center announced USA 2024",
    "Meta data center announced USA 2024",
]


def _now_utc_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def _as_null_if_blank(x: Any) -> Any:
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, str) and x.strip() == "":
        return None
    return x


def _norm_key(x: Any) -> str:
    x = _as_null_if_blank(x)
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x).strip().lower())


def _valid_url(url: Any) -> bool:
    if not isinstance(url, str):
        return False
    u = url.strip()
    return u.startswith("http://") or u.startswith("https://")


def _write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def _read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    out: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return out


def _dedupe_dicts_by_key(rows: List[Dict[str, Any]], key: str) -> List[Dict[str, Any]]:
    seen: set[str] = set()
    out: List[Dict[str, Any]] = []
    for r in rows:
        v = r.get(key)
        if not isinstance(v, str) or not v.strip():
            continue
        if v in seen:
            continue
        seen.add(v)
        out.append(r)
    return out


def serpapi_search(api_key: str, query: str, num: int = 10) -> List[Dict[str, Any]]:
    resp = requests.get(
        "https://serpapi.com/search.json",
        params={"engine": "google", "q": query, "api_key": api_key, "num": num},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    out: List[Dict[str, Any]] = []
    for item in data.get("organic_results", []) or []:
        out.append(
            {
                "title": item.get("title"),
                "url": item.get("link"),
                "snippet": item.get("snippet"),
                "query": query,
                "search_provider": "serpapi",
            }
        )
    return out


def search_and_collect_articles(queries: List[str], *, num_results_per_query: int = 10, sleep_s: float = 1.0) -> List[Dict[str, Any]]:
    api_key = os.environ.get("SERPAPI_API_KEY")
    if not api_key:
        raise RuntimeError("SERPAPI_API_KEY is required.")

    results: List[Dict[str, Any]] = []
    for q in queries:
        results.extend(serpapi_search(api_key, q, num=num_results_per_query))
        time.sleep(max(0.0, sleep_s))

    seen = set()
    deduped: List[Dict[str, Any]] = []
    for r in results:
        url = (r.get("url") or "").strip()
        if not url or url in seen:
            continue
        seen.add(url)
        deduped.append(r)
    return deduped


def scrape_article_text(url: str) -> Dict[str, Any]:
    if not _valid_url(url):
        return {"url": url, "text": None, "scrape_method": None, "scrape_error": "invalid_url"}

    if Article is not None:
        try:
            a = Article(url)
            a.download()
            a.parse()
            text = (a.text or "").strip() or None
            return {"url": url, "text": text, "scrape_method": "newspaper3k", "scrape_error": None}
        except Exception as e:
            newspaper_err = f"{type(e).__name__}: {e}"
    else:
        newspaper_err = "newspaper3k_unavailable"

    try:
        resp = requests.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/123 Safari/537.36"
            },
            timeout=30,
        )
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()
        text = soup.get_text("\n")
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]{2,}", " ", text)
        text = text.strip() or None
        return {"url": url, "text": text, "scrape_method": "beautifulsoup", "scrape_error": None, "fallback_from": newspaper_err}
    except Exception as e:
        return {"url": url, "text": None, "scrape_method": "beautifulsoup", "scrape_error": f"{type(e).__name__}: {e}", "fallback_from": newspaper_err}


def _state_normalize_to_abbrev(x: Any) -> Optional[str]:
    x = _as_null_if_blank(x)
    if x is None:
        return None
    s = str(x).strip()
    if len(s) == 2 and s.upper() in US_STATE_ABBREV:
        return s.upper()
    return US_STATE_NAME_TO_ABBREV.get(_norm_key(s))


def _year_parse(x: Any) -> Optional[int]:
    x = _as_null_if_blank(x)
    if x is None:
        return None
    if isinstance(x, int):
        return x
    if isinstance(x, float) and x.is_integer():
        return int(x)
    s = str(x).strip()
    if re.fullmatch(r"\d{4}", s):
        return int(s)
    return None


def _status_normalize(x: Any) -> Optional[Status]:
    x = _as_null_if_blank(x)
    if x is None:
        return None
    s = _norm_key(x)
    mapping = {
        "announced": "announced",
        "announcement": "announced",
        "under_construction": "under_construction",
        "under construction": "under_construction",
        "construction": "under_construction",
        "opened": "opened",
        "open": "opened",
        "operational": "opened",
        "operating": "opened",
    }
    return mapping.get(s)  # type: ignore[return-value]


def openai_extract_strict(*, article_text: str, source_url: str, title: Optional[str], snippet: Optional[str]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        return (
            {"company": None, "city": None, "state": None, "year_announced": None, "status": None, "investment_usd": None, "source_url": source_url},
            {"extraction_method": None, "extraction_error": "OPENAI_API_KEY_not_set"},
        )

    model = os.environ.get("OPENAI_MODEL", "gpt-4.1")
    fallback_model = os.environ.get("OPENAI_FALLBACK_MODEL", "gpt-4.1")

    system = (
        "You extract structured facts from a news article about U.S. data center developments.\n"
        "Return ONLY valid JSON matching the given schema exactly.\n\n"
        "Hard rules (no exceptions):\n"
        "- If a field is not explicitly stated in the text, output null.\n"
        "- Do NOT infer, assume, approximate, or 'fill in' missing details.\n"
        "- Do NOT add extra keys.\n"
        "- Always include all keys shown in the schema.\n"
        "- Always keep source_url exactly as provided.\n"
        "- status must be one of: announced, under_construction, opened, or null.\n"
        "- If the text gives a full U.S. state name, convert it to the 2-letter USPS abbreviation.\n"
        "- investment_usd: only if an explicit numeric amount is stated (otherwise null).\n"
    )

    schema = {
        "company": None,
        "city": None,
        "state": None,
        "year_announced": None,
        "status": None,
        "investment_usd": None,
        "source_url": source_url,
    }
    user = {
        "task": "Extract ONLY explicitly stated info into the exact schema below.",
        "schema": schema,
        "allowed_status_values": ["announced", "under_construction", "opened", None],
        "article": {"title": title, "snippet": snippet, "source_url": source_url, "text": article_text},
    }

    url = "https://api.openai.com/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

    def post(payload: Dict[str, Any]) -> requests.Response:
        return requests.post(url, headers=headers, json=payload, timeout=60)

    base_payload: Dict[str, Any] = {
        "model": model,
        "temperature": 0,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": json.dumps(user)},
        ],
    }

    resp = post({**base_payload, "response_format": {"type": "json_object"}})
    if resp.status_code >= 400:
        resp2 = post(base_payload)
        if resp2.status_code < 400:
            resp = resp2
        else:
            payload3 = {**base_payload, "model": fallback_model}
            resp3 = post({**payload3, "response_format": {"type": "json_object"}})
            resp = resp3 if resp3.status_code < 400 else post(payload3)

    try:
        resp.raise_for_status()
        data = resp.json()
        content = data["choices"][0]["message"]["content"]
        if content.strip().startswith("{"):
            parsed = json.loads(content)
        else:
            m = re.search(r"\{[\s\S]*\}\s*$", content.strip())
            parsed = json.loads(m.group(0)) if m else {}
        row = {k: parsed.get(k, None) for k in schema.keys()}
        row["source_url"] = source_url
        row["state"] = _state_normalize_to_abbrev(row.get("state"))
        row["year_announced"] = _year_parse(row.get("year_announced"))
        row["status"] = _status_normalize(row.get("status"))
        return row, {"extraction_method": "openai", "model": model, "extraction_error": None}
    except Exception as e:
        base = {k: None for k in schema.keys()}
        base["source_url"] = source_url
        return base, {"extraction_method": "openai", "model": model, "extraction_error": f"{type(e).__name__}: {e}"}


def _missing_key_fields(row: Dict[str, Any]) -> int:
    key_fields = ["company", "city", "state", "year_announced", "status"]
    return sum(_as_null_if_blank(row.get(k)) is None for k in key_fields)


def _dedupe_choose_best(group: pd.DataFrame) -> pd.Series:
    def score_row(r: pd.Series) -> Tuple[int, int]:
        key_fields = ["company", "city", "state", "year_announced", "status", "investment_usd"]
        non_null = sum(_as_null_if_blank(r.get(k)) is not None for k in key_fields)
        has_url = 1 if _valid_url(r.get("source_url")) else 0
        return non_null, has_url

    scored = group.copy()
    scored["_score"] = scored.apply(score_row, axis=1)
    scored = scored.sort_values(by="_score", ascending=False)
    return scored.iloc[0].drop(labels=["_score"])


def combine_dedupe_filter(extracted: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    combined = extracted.copy()

    combined["company"] = combined["company"].apply(_as_null_if_blank)
    combined["city"] = combined["city"].apply(_as_null_if_blank)
    combined["state"] = combined["state"].apply(_state_normalize_to_abbrev)
    combined["year_announced"] = combined["year_announced"].apply(_year_parse)
    combined["status"] = combined["status"].apply(_status_normalize)
    combined["investment_usd"] = combined["investment_usd"].apply(_as_null_if_blank)
    combined["source_url"] = combined["source_url"].apply(_as_null_if_blank)

    combined["_k_company"] = combined["company"].apply(_norm_key)
    combined["_k_city"] = combined["city"].apply(_norm_key)
    combined["_k_state"] = combined["state"].apply(_norm_key)

    deduped = (
        combined.groupby(["_k_company", "_k_city", "_k_state"], dropna=False, sort=False)
        .apply(_dedupe_choose_best, include_groups=False)
        .reset_index(drop=True)
    )

    dropped_reasons: List[str] = []
    keep_mask: List[bool] = []
    for _, r in deduped.iterrows():
        reasons: List[str] = []
        if _as_null_if_blank(r.get("company")) is None:
            reasons.append("missing_company")
        if _as_null_if_blank(r.get("state")) is None:
            reasons.append("missing_state")
        year = _year_parse(r.get("year_announced"))
        if year is not None and not (YEAR_MIN <= year <= YEAR_MAX):
            reasons.append("year_out_of_range")
        if _missing_key_fields(r.to_dict()) > 2:
            reasons.append("missing_more_than_2_key_fields")
        keep = len(reasons) == 0
        keep_mask.append(keep)
        dropped_reasons.append(";".join(reasons) if reasons else "")

    deduped["_drop_reasons"] = dropped_reasons
    final_df = deduped[pd.Series(keep_mask, index=deduped.index)].copy()
    dropped_df = deduped[~pd.Series(keep_mask, index=deduped.index)].copy()

    for df in (final_df, dropped_df):
        df.drop(columns=[c for c in df.columns if c.startswith("_k_")], inplace=True, errors="ignore")

    col_order = ["company","city","state","year_announced","status","investment_usd","source_url","source_type","extraction_confidence"]
    final_df = final_df[[c for c in col_order if c in final_df.columns]].copy()
    dropped_df = dropped_df[[c for c in (col_order + ["_drop_reasons","_extraction_meta"]) if c in dropped_df.columns]].copy()
    return final_df, dropped_df


def _confidence_for_row(row: Dict[str, Any]) -> str:
    key_fields = ["company", "city", "state", "year_announced", "status"]
    non_null = sum(_as_null_if_blank(row.get(k)) is not None for k in key_fields)
    has_url = _valid_url(row.get("source_url"))
    if has_url and non_null >= 5:
        return "high"
    if has_url and non_null >= 3:
        return "medium"
    return "low"


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="data", help="Output directory (default: data)")
    parser.add_argument("--num", type=int, default=10, help="Results per query")
    parser.add_argument("--sleep", type=float, default=1.0, help="Delay between search queries")
    parser.add_argument("--scrape-delay", type=float, default=1.5, help="Polite delay between URL scrapes")
    parser.add_argument("--scrape-jitter", type=float, default=0.5, help="Random jitter added to scrape delay")
    parser.add_argument("--append", action="store_true", help="Cumulative mode (don’t re-scrape/re-extract known URLs)")
    parser.add_argument("--from-raw", action="store_true", help="Skip search/scrape; use existing raw_articles.jsonl")
    parser.add_argument("--queries", nargs="*", default=None, help="Override queries")
    args = parser.parse_args()

    data_dir = Path(args.data_dir)
    raw_path = data_dir / "raw_articles.jsonl"
    dropped_path = data_dir / "dropped_rows.jsonl"
    out_path = data_dir / "new_data_centers.csv"

    prior_raw = _read_jsonl(raw_path) if args.append else []
    prior_raw_urls = {r.get("url") for r in prior_raw if isinstance(r.get("url"), str)}

    prior_extracted_urls: set[str] = set()
    if args.append and out_path.exists():
        try:
            prev = pd.read_csv(out_path)
            if "source_url" in prev.columns:
                prior_extracted_urls = set(prev["source_url"].dropna().astype(str).tolist())
        except Exception:
            pass

    if args.from_raw:
        raw_rows = _read_jsonl(raw_path)
        if not raw_rows:
            raise RuntimeError(f"{raw_path} is missing/empty but --from-raw was set.")
    else:
        queries = args.queries if args.queries else DEFAULT_QUERIES
        search_results = search_and_collect_articles(queries, num_results_per_query=args.num, sleep_s=args.sleep)

        raw_new: List[Dict[str, Any]] = []
        for r in search_results:
            url = (r.get("url") or "").strip()
            if not url:
                continue
            if args.append and url in prior_raw_urls:
                continue
            time.sleep(max(0.0, args.scrape_delay) + random.random() * max(0.0, args.scrape_jitter))
            scraped = scrape_article_text(url)
            raw_new.append(
                {
                    "collected_at_utc": _now_utc_iso(),
                    "title": r.get("title"),
                    "url": url,
                    "snippet": r.get("snippet"),
                    "query": r.get("query"),
                    "search_provider": "serpapi",
                    "scrape_method": scraped.get("scrape_method"),
                    "scrape_error": scraped.get("scrape_error"),
                    "text": scraped.get("text"),
                }
            )

        raw_rows = prior_raw + raw_new if args.append else raw_new
        raw_rows = _dedupe_dicts_by_key(raw_rows, "url")
        _write_jsonl(raw_path, raw_rows)

    extracted_rows: List[Dict[str, Any]] = []
    for raw in raw_rows:
        url = str(raw.get("url") or "").strip()
        if args.append and url and url in prior_extracted_urls:
            continue
        text = raw.get("text")
        if not isinstance(text, str) or not text.strip():
            extracted = {"company": None, "city": None, "state": None, "year_announced": None, "status": None, "investment_usd": None, "source_url": url}
            meta = {"extraction_method": None, "extraction_error": "missing_article_text"}
        else:
            extracted, meta = openai_extract_strict(article_text=text, source_url=url, title=raw.get("title"), snippet=raw.get("snippet"))
        extracted["source_type"] = "extracted"
        extracted["extraction_confidence"] = _confidence_for_row(extracted)
        extracted["_extraction_meta"] = meta
        extracted_rows.append(extracted)

    extracted_df = pd.DataFrame(extracted_rows)

    if args.append and out_path.exists():
        prev = pd.read_csv(out_path)
        extracted_df = pd.concat([prev, extracted_df], ignore_index=True)
        extracted_df = extracted_df.drop_duplicates(subset=["source_url"], keep="first")

    final_df, dropped_df = combine_dedupe_filter(extracted_df)

    data_dir.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(out_path, index=False, quoting=csv.QUOTE_MINIMAL)
    _write_jsonl(dropped_path, (r for r in dropped_df.to_dict(orient="records")))

    print(f"Wrote {raw_path}")
    print(f"Wrote {dropped_path}")
    print(f"Wrote {out_path} (rows={len(final_df)})")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())